In [6]:
!pip install spacy
!python -m spacy download en_core_web_md

  Using cached typer-0.12.5-py3-none-any.whl.metadata (15 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 506.9 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.1/839.1 kB 543.5 kB/s eta 0:00:00a 0:00:01
Using cached typer-0.12.5-py3-none-any.whl (47 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 687.9 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 598.5 kB/s eta 0:00:00a 0:00:01
Using cached shellingham-1.5.4-py2.py3-none-any.whl (9.8 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 611.1 kB/s eta 0:00:0000:0100:03
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')


In [38]:
!pip install fuzzywuzzy

  Using cached fuzzywuzzy-0.18.0-py2.py3-none-any.whl.metadata (4.9 kB)
Using cached fuzzywuzzy-0.18.0-py2.py3-none-any.whl (18 kB)


In [18]:
import pandas as pd

# Load the Excel file into a dataframe
file_path = '/Users/shravyadsouza/Desktop/MIT/DE-minipro-ingredients.xlsx'  # Adjust this path as needed
df = pd.read_excel(file_path)

# Display the first few rows to confirm the dataframe is loaded
df.head()

,Ingredient,Restricted Form,Preferred Form,Frequency,Vatta,Pitta,Kapha,Fruits,Vegetables,Grains,...,Animal Foods,Condiments,Nuts,Seeds,Oils,Beverages,Herbal Teas,Spices,Sweeteners,Food Supplements
0,Aduki Beans,All,NaN,NaN,1,0,0,0.0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,Aduki Beans,NaN,All,NaN,0,1,1,0.0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,Ajwan,NaN,All,NaN,1,0,0,0.0,0,0,...,0,0,0,0,0,0,1,0,0,0
3,Ajwan,All,NaN,NaN,0,1,0,0.0,0,0,...,0,0,0,0,0,0,1,0,0,0
4,Ajwan,NaN,All,NaN,1,0,1,0.0,0,0,...,0,0,0,0,0,0,1,1,0,0


In [8]:
import spacy

# Load the medium-sized pre-trained English model
nlp = spacy.load("en_core_web_md")

In [22]:
def get_ingredient_vector(ingredient_name):
    """Convert an ingredient name to a vector using SpaCy."""
    return nlp(ingredient_name).vector

In [24]:
def calculate_similarity(ingredient1, ingredient2):
    """Calculate the similarity between two ingredients using their SpaCy vectors."""
    # Convert ingredient names to vectors
    vector1 = nlp(ingredient1).vector
    vector2 = nlp(ingredient2).vector
    
    # Calculate cosine similarity between vectors
    similarity = nlp(ingredient1).similarity(nlp(ingredient2))
    return similarity

In [26]:
def filter_by_category_and_dosha(df, category, dosha_type):
    """
    Filter the DataFrame to include only ingredients that belong to the specified category
    and are suitable for the given dosha type.
    """
    if category not in df.columns or dosha_type not in df.columns:
        print(f"Category '{category}' or Dosha '{dosha_type}' not found in DataFrame.")
        return pd.DataFrame()

    # Filter ingredients based on the category (category column value should be 1)
    category_filtered_df = df[df[category] == 1]

    # Further filter based on the dosha type (dosha column value should be 1)
    dosha_filtered_df = category_filtered_df[category_filtered_df[dosha_type] == 1]

    return dosha_filtered_df

In [28]:
def find_most_similar_ingredients(df, target_ingredient, category, dosha_type, top_n=5):
    """
    Find the top N most similar ingredients to a given ingredient based on category and dosha type.
    """
    # Step 1: Filter the DataFrame based on category and dosha type
    filtered_df = filter_by_category_and_dosha(df, category, dosha_type)

    if filtered_df.empty:
        print(f"No ingredients found for category '{category}' and dosha type '{dosha_type}'.")
        return []

    # Step 2: Calculate similarity between the target ingredient and filtered ingredients
    similarities = []
    for ingredient in filtered_df['Ingredient'].unique():
        similarity_score = calculate_similarity(target_ingredient, ingredient)
        similarities.append((ingredient, similarity_score))

    # Step 3: Sort ingredients based on similarity score and return the top N
    similarities = sorted(similarities, key=lambda x: x[1], reverse=True)
    return similarities[:top_n]

In [32]:
# Example usage: Find the top 5 most similar ingredients to "Soya Milk" for category "Dairy" and dosha type "Pitta"
similar_ingredients = find_most_similar_ingredients(
    df, 
    "Soya Milk",         # Target ingredient
    category="Dairy",    # Category to filter by
    dosha_type="Pitta",  # Dosha type to filter by
    top_n=5              # Number of similar ingredients to return
)

# Display the similar ingredients
similar_ingredients

/var/folders/4z/s5y79j750db8zw6k37sy40_w0000gn/T/ipykernel_63000/3481395619.py:8: UserWarning: [W008] Evaluating Doc.similarity based on empty vectors.
  similarity = nlp(ingredient1).similarity(nlp(ingredient2))


[('Ghee', 1.0000000509849807),
 ('Butter', 0.6817293845420794),
 ('Buttermilk', 0.6817293845420794),
 ('Cheese', 0.5665353052871895),
 ("Cow's Milk", 0.5518949958280054)]

In [34]:
import spacy

# Load the medium-sized pre-trained English model
nlp = spacy.load("en_core_web_md")

def calculate_similarity(ingredient1, ingredient2):
    """Calculate the similarity between two ingredients using their SpaCy vectors."""
    return nlp(ingredient1).similarity(nlp(ingredient2))

def filter_by_category_and_dosha(df, category, dosha_type):
    """
    Filter the DataFrame to include only ingredients that belong to the specified category
    and are suitable for the given dosha type.
    """
    if category not in df.columns or dosha_type not in df.columns:
        print(f"Category '{category}' or Dosha '{dosha_type}' not found in DataFrame.")
        return pd.DataFrame()

    # Filter ingredients based on the category (category column value should be 1)
    category_filtered_df = df[df[category] == 1]

    # Further filter based on the dosha type (dosha column value should be 1)
    dosha_filtered_df = category_filtered_df[category_filtered_df[dosha_type] == 1]

    return dosha_filtered_df

def find_most_similar_ingredients(df, target_ingredient, category, dosha_type, top_n=5, similarity_threshold=0.6):
    """
    Find the top N most similar ingredients to a given ingredient based on category and dosha type.
    Enforce a similarity threshold to filter out low-similarity results.
    """
    # Step 1: Filter the DataFrame based on category and dosha type
    filtered_df = filter_by_category_and_dosha(df, category, dosha_type)

    if filtered_df.empty:
        print(f"No ingredients found for category '{category}' and dosha type '{dosha_type}'.")
        return []

    # Step 2: Calculate similarity between the target ingredient and filtered ingredients
    similarities = []
    for ingredient in filtered_df['Ingredient'].unique():
        similarity_score = calculate_similarity(target_ingredient, ingredient)
        
        # Only consider ingredients with similarity above the threshold
        if similarity_score >= similarity_threshold:
            similarities.append((ingredient, similarity_score))

    # Step 3: Sort ingredients based on similarity score and return the top N
    similarities = sorted(similarities, key=lambda x: x[1], reverse=True)
    
    # Print information for debugging
    print(f"Filtered and valid alternatives for '{target_ingredient}' (Category: {category}, Dosha: {dosha_type}):")
    for item in similarities:
        print(item)

    return similarities[:top_n]

# Example usage: Find the top 5 most similar ingredients to "Soy Milk" for category "Dairy" and dosha type "Pitta"
similar_ingredients = find_most_similar_ingredients(
    df, 
    "Soy Milk",         # Target ingredient
    category="Dairy",    # Category to filter by
    dosha_type="Pitta",  # Dosha type to filter by
    top_n=5,             # Number of similar ingredients to return
    similarity_threshold=0.6  # Similarity threshold to filter out low-similarity results
)

# Display the similar ingredients
similar_ingredients


Filtered and valid alternatives for 'Soy Milk' (Category: Dairy, Dosha: Pitta):
('Ghee', 0.8167307547960709)
('Butter', 0.6068263068334911)
('Buttermilk', 0.6068263068334911)


/var/folders/4z/s5y79j750db8zw6k37sy40_w0000gn/T/ipykernel_63000/1529706169.py:8: UserWarning: [W008] Evaluating Doc.similarity based on empty vectors.
  return nlp(ingredient1).similarity(nlp(ingredient2))


[('Ghee', 0.8167307547960709),
 ('Butter', 0.6068263068334911),
 ('Buttermilk', 0.6068263068334911)]

In [42]:
from fuzzywuzzy import process
import spacy

# Load the medium-sized pre-trained English model
nlp = spacy.load("en_core_web_md")

def calculate_similarity(ingredient1, ingredient2):
    """Calculate the similarity between two ingredients using their SpaCy vectors."""
    return nlp(ingredient1).similarity(nlp(ingredient2))

def filter_by_category_and_dosha(df, category, dosha_type):
    """
    Filter the DataFrame to include only ingredients that belong to the specified category
    and are suitable for the given dosha type.
    """
    if category not in df.columns or dosha_type not in df.columns:
        print(f"Category '{category}' or Dosha '{dosha_type}' not found in DataFrame.")
        return pd.DataFrame()

    # Filter ingredients based on the category (category column value should be 1)
    category_filtered_df = df[df[category] == 1]

    # Further filter based on the dosha type (dosha column value should be 1)
    dosha_filtered_df = category_filtered_df[category_filtered_df[dosha_type] == 1]

    return dosha_filtered_df

def fuzzy_match_ingredients(ingredient, all_ingredients, limit=5, cutoff=60):
    """
    Use fuzzy matching to identify the top ingredients that match the given ingredient.
    Returns a list of ingredient names with their fuzzy match scores.
    """
    fuzzy_matches = process.extract(ingredient, all_ingredients, limit=limit, scorer=process.fuzz.ratio)
    return [(match[0], match[1]) for match in fuzzy_matches if match[1] >= cutoff]

def find_hybrid_similar_ingredients(df, target_ingredient, category, dosha_type, top_n=5, similarity_threshold=0.3, fuzzy_cutoff=60):
    """
    Find the top N most similar ingredients using a hybrid approach that combines fuzzy matching and semantic similarity.
    Lower the similarity threshold to ensure that fuzzy matches are prioritized.
    """
    # Step 1: Filter the DataFrame based on category and dosha type
    filtered_df = filter_by_category_and_dosha(df, category, dosha_type)

    if filtered_df.empty:
        print(f"No ingredients found for category '{category}' and dosha type '{dosha_type}'.")
        return []

    # Step 2: Fuzzy match the target ingredient against the filtered ingredients
    all_ingredients = filtered_df['Ingredient'].tolist()
    fuzzy_matches = fuzzy_match_ingredients(target_ingredient, all_ingredients, limit=10, cutoff=fuzzy_cutoff)

    # Print fuzzy matching results for debugging
    print(f"Fuzzy matching results for '{target_ingredient}': {fuzzy_matches}")

    # Step 3: Calculate semantic similarity for the fuzzy matches
    similarities = []
    for fuzzy_match in fuzzy_matches:
        ingredient_name, fuzzy_score = fuzzy_match
        similarity_score = calculate_similarity(target_ingredient, ingredient_name)

        # Only consider ingredients with a similarity above a reduced threshold
        if similarity_score >= similarity_threshold:
            # Prioritize fuzzy score by combining with similarity score
            combined_score = (similarity_score + fuzzy_score / 100) / 2  # Adjust weight as needed
            similarities.append((ingredient_name, combined_score))

    # Step 4: Sort combined scores to get the top N alternatives
    similarities = sorted(similarities, key=lambda x: x[1], reverse=True)

    # Print information for debugging
    print(f"Filtered and valid alternatives for '{target_ingredient}' (Category: {category}, Dosha: {dosha_type}):")
    for item in similarities:
        print(item)

    return similarities[:top_n]

# Example usage: Find the top 5 most similar ingredients to "Soy Milk" for category "Dairy" and dosha type "Pitta"
similar_ingredients = find_hybrid_similar_ingredients(
    df, 
    "Soy Milk",         # Target ingredient
    category="Dairy",    # Category to filter by
    dosha_type="Pitta",  # Dosha type to filter by
    top_n=5,             # Number of similar ingredients to return
    similarity_threshold=0.3,  # Reduced similarity threshold to include fuzzy matches
    fuzzy_cutoff=60     # Fuzzy match cutoff score to consider as valid match
)

# Display the similar ingredients
similar_ingredients


Fuzzy matching results for 'Soy Milk': [("Cow's Milk", 67), ("Goat's Milk", 63)]
Filtered and valid alternatives for 'Soy Milk' (Category: Dairy, Dosha: Pitta):
("Cow's Milk", 0.5378852421972035)
("Goat's Milk", 0.5178852421972034)


[("Cow's Milk", 0.5378852421972035), ("Goat's Milk", 0.5178852421972034)]

In [46]:
from fuzzywuzzy import process
import spacy

# Load the medium-sized pre-trained English model
nlp = spacy.load("en_core_web_md")

def calculate_similarity(ingredient1, ingredient2):
    """Calculate the similarity between two ingredients using their SpaCy vectors."""
    return nlp(ingredient1).similarity(nlp(ingredient2))

def filter_by_category_and_dosha(df, category, dosha_type):
    """
    Filter the DataFrame to include only ingredients that belong to the specified category
    and are suitable for the given dosha type.
    """
    if category not in df.columns or dosha_type not in df.columns:
        print(f"Category '{category}' or Dosha '{dosha_type}' not found in DataFrame.")
        return pd.DataFrame()

    # Filter ingredients based on the category (category column value should be 1)
    category_filtered_df = df[df[category] == 1]

    # Further filter based on the dosha type (dosha column value should be 1)
    dosha_filtered_df = category_filtered_df[category_filtered_df[dosha_type] == 1]

    return dosha_filtered_df

def fuzzy_match_ingredients(ingredient, all_ingredients, limit=10, cutoff=50):
    """
    Use fuzzy matching to identify the top ingredients that match the given ingredient.
    Returns a list of ingredient names with their fuzzy match scores.
    """
    fuzzy_matches = process.extract(ingredient, all_ingredients, limit=limit, scorer=process.fuzz.ratio)
    return [(match[0], match[1]) for match in fuzzy_matches if match[1] >= cutoff]

def find_hybrid_similar_ingredients(df, target_ingredient, category, dosha_type, top_n=5, similarity_threshold=0.2, fuzzy_cutoff=50, fuzzy_weight=0.7):
    """
    Find the top N most similar ingredients using a hybrid approach that combines fuzzy matching and semantic similarity.
    - Lower similarity and fuzzy thresholds to include more alternatives.
    - Use weighted combination of fuzzy and semantic scores.
    - `fuzzy_weight` determines how much importance is given to fuzzy scores (between 0 and 1).
    """
    # Step 1: Filter the DataFrame based on category and dosha type
    filtered_df = filter_by_category_and_dosha(df, category, dosha_type)

    if filtered_df.empty:
        print(f"No ingredients found for category '{category}' and dosha type '{dosha_type}'.")
        return []

    # Step 2: Fuzzy match the target ingredient against the filtered ingredients
    all_ingredients = filtered_df['Ingredient'].tolist()
    fuzzy_matches = fuzzy_match_ingredients(target_ingredient, all_ingredients, limit=10, cutoff=fuzzy_cutoff)

    # Print fuzzy matching results for debugging
    print(f"Fuzzy matching results for '{target_ingredient}': {fuzzy_matches}")

    # Step 3: Calculate semantic similarity for the fuzzy matches
    similarities = []
    for fuzzy_match in fuzzy_matches:
        ingredient_name, fuzzy_score = fuzzy_match
        similarity_score = calculate_similarity(target_ingredient, ingredient_name)

        # Include all fuzzy matches irrespective of semantic similarity (relax this condition)
        if similarity_score >= similarity_threshold or fuzzy_score > fuzzy_cutoff:
            # Prioritize fuzzy score by using weighted combination with similarity score
            combined_score = fuzzy_weight * (fuzzy_score / 100) + (1 - fuzzy_weight) * similarity_score
            similarities.append((ingredient_name, combined_score))

    # Step 4: Sort combined scores to get the top N alternatives
    similarities = sorted(similarities, key=lambda x: x[1], reverse=True)

    # Print information for debugging
    print(f"Filtered and valid alternatives for '{target_ingredient}' (Category: {category}, Dosha: {dosha_type}):")
    for item in similarities:
        print(item)

    return similarities[:top_n]

# Example usage: Find the top 5 most similar ingredients to "Soy Milk" for category "Dairy" and dosha type "Pitta"
similar_ingredients = find_hybrid_similar_ingredients(
    df, 
    "Burdock root",         # Target ingredient
    category="Vegetables",    # Category to filter by
    dosha_type="Pitta",  # Dosha type to filter by
    top_n=5,             # Number of similar ingredients to return
    similarity_threshold=0.2,  # Further reduced similarity threshold to include more alternatives
    fuzzy_cutoff=50,     # Further reduced fuzzy cutoff score
    fuzzy_weight=0.7     # Give more weight to fuzzy scores over semantic similarity
)

# Display the similar ingredients
similar_ingredients


Fuzzy matching results for 'Burdock root': [('Burdock root', 100), ('Taro Root', 67), ('Broccoli', 50)]
Filtered and valid alternatives for 'Burdock root' (Category: Vegetables, Dosha: Pitta):
('Burdock root', 1.0)
('Taro Root', 0.5946746369736429)
('Broccoli', 0.41378099845870264)


[('Burdock root', 1.0),
 ('Taro Root', 0.5946746369736429),
 ('Broccoli', 0.41378099845870264)]

In [48]:
from fuzzywuzzy import process
import spacy
import re

# Load the medium-sized pre-trained English model
nlp = spacy.load("en_core_web_md")

def calculate_similarity(ingredient1, ingredient2):
    """Calculate the similarity between two ingredients using their SpaCy vectors."""
    return nlp(ingredient1).similarity(nlp(ingredient2))

def filter_by_category_and_dosha(df, category, dosha_type):
    """
    Filter the DataFrame to include only ingredients that belong to the specified category
    and are suitable for the given dosha type.
    """
    if category not in df.columns or dosha_type not in df.columns:
        print(f"Category '{category}' or Dosha '{dosha_type}' not found in DataFrame.")
        return pd.DataFrame()

    # Filter ingredients based on the category (category column value should be 1)
    category_filtered_df = df[df[category] == 1]

    # Further filter based on the dosha type (dosha column value should be 1)
    dosha_filtered_df = category_filtered_df[category_filtered_df[dosha_type] == 1]

    return dosha_filtered_df

def fuzzy_match_ingredients(ingredient, all_ingredients, limit=10, cutoff=50):
    """
    Use fuzzy matching to identify the top ingredients that match the given ingredient.
    Returns a list of ingredient names with their fuzzy match scores.
    """
    fuzzy_matches = process.extract(ingredient, all_ingredients, limit=limit, scorer=process.fuzz.ratio)
    return [(match[0], match[1]) for match in fuzzy_matches if match[1] >= cutoff]

def find_hybrid_similar_ingredients(df, target_ingredient, category, dosha_type, top_n=5, similarity_threshold=0.2, fuzzy_cutoff=50, fuzzy_weight=0.7):
    """
    Find the top N most similar ingredients using a hybrid approach that combines fuzzy matching and semantic similarity.
    Exclude self-matching ingredients from the results.
    - `fuzzy_weight` determines how much importance is given to fuzzy scores (between 0 and 1).
    """
    # Step 1: Filter the DataFrame based on category and dosha type
    filtered_df = filter_by_category_and_dosha(df, category, dosha_type)

    if filtered_df.empty:
        print(f"No ingredients found for category '{category}' and dosha type '{dosha_type}'.")
        return []

    # Step 2: Fuzzy match the target ingredient against the filtered ingredients
    all_ingredients = filtered_df['Ingredient'].tolist()
    fuzzy_matches = fuzzy_match_ingredients(target_ingredient, all_ingredients, limit=10, cutoff=fuzzy_cutoff)

    # Print fuzzy matching results for debugging
    print(f"Fuzzy matching results for '{target_ingredient}': {fuzzy_matches}")

    # Step 3: Calculate semantic similarity for the fuzzy matches
    similarities = []
    for fuzzy_match in fuzzy_matches:
        ingredient_name, fuzzy_score = fuzzy_match
        similarity_score = calculate_similarity(target_ingredient, ingredient_name)

        # Exclude the ingredient if it matches the target ingredient exactly (or via regex pattern)
        if re.fullmatch(re.escape(target_ingredient.lower()), ingredient_name.lower()):
            print(f"Excluding self-match: '{ingredient_name}' for target '{target_ingredient}'")
            continue

        # Include all fuzzy matches irrespective of semantic similarity (relax this condition)
        if similarity_score >= similarity_threshold or fuzzy_score > fuzzy_cutoff:
            # Prioritize fuzzy score by using weighted combination with similarity score
            combined_score = fuzzy_weight * (fuzzy_score / 100) + (1 - fuzzy_weight) * similarity_score
            similarities.append((ingredient_name, combined_score))

    # Step 4: Sort combined scores to get the top N alternatives
    similarities = sorted(similarities, key=lambda x: x[1], reverse=True)

    # Print information for debugging
    print(f"Filtered and valid alternatives for '{target_ingredient}' (Category: {category}, Dosha: {dosha_type}):")
    for item in similarities:
        print(item)

    return similarities[:top_n]

# Example usage: Find the top 5 most similar ingredients to "Burdock root" for category "Vegetables" and dosha type "Pitta"
similar_ingredients = find_hybrid_similar_ingredients(
    df, 
    "Burdock root",      # Target ingredient
    category="Vegetables",  # Category to filter by
    dosha_type="Pitta",  # Dosha type to filter by
    top_n=5,             # Number of similar ingredients to return
    similarity_threshold=0.2,  # Reduced similarity threshold to include more alternatives
    fuzzy_cutoff=50,     # Further reduced fuzzy cutoff score
    fuzzy_weight=0.7     # Give more weight to fuzzy scores over semantic similarity
)

# Display the similar ingredients
similar_ingredients


Fuzzy matching results for 'Burdock root': [('Burdock root', 100), ('Taro Root', 67), ('Broccoli', 50)]
Excluding self-match: 'Burdock root' for target 'Burdock root'
Filtered and valid alternatives for 'Burdock root' (Category: Vegetables, Dosha: Pitta):
('Taro Root', 0.5946746369736429)
('Broccoli', 0.41378099845870264)


[('Taro Root', 0.5946746369736429), ('Broccoli', 0.41378099845870264)]